# 0.27 — Thematic-ETF keyword **time series** (6 themes)

Six themes chosen from `data/input/Thematic_ETF_search.csv`: **space · drones · crypto · e-sports ·
online retail · cyber**. For each theme a hand-built validation lexicon (annotation only — the same
logic as `0.11`/`0.21`), monthly counts of matching Bloomberg headlines from **the preprocessed
corpus `data/processed/news_corpus.parquet`** (2010–2025, ADD-event dating — replay artifact proven
in 0.25, corpus built/certified in 0.26), overlaid with **ETF inception dates from the CSV (red)**
and **externally verified event milestones (gray)**.

Web-verified milestones: Falcon Heavy **2018-02-06** · Virgin Galactic listing **2019-10-28** ·
Crew Dragon Demo-2 **2020-05-30** · Starship first flight **2023-04-20** · Amazon Prime Air reveal
**2013-12-01** · FAA Part 107 **2016-08-29** · Gatwick drone shutdown **2018-12-19** · Joby NYSE
debut **2021-08-11** · CME bitcoin futures **2017-12-17** · Coinbase IPO **2021-04-14** · FTX
bankruptcy **2022-11-11** · spot-BTC ETFs approved **2024-01-10** · Amazon buys Twitch
**2014-08-25** · Fortnite Battle Royale **2017-09-26** · Microsoft bids for Activision
**2022-01-18** · Alibaba IPO **2014-09-19** · Amazon buys Whole Foods **2017-06-16** · WannaCry
**2017-05-12** · Equifax breach **2017-09-07** · SolarWinds hack **2020-12-13** · Colonial Pipeline
**2021-05-07** · CrowdStrike outage **2024-07-19**.

In [ ]:
import re
from pathlib import Path
import numpy as np, pandas as pd
import plotly.graph_objects as go
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NB_DIR = _ROOT / "notebooks"; OUTPUT_DIR = NB_DIR / "output"

# --- ETF universe: tickers + inception dates straight from the screener export ---
etf = pd.read_csv(_ROOT / "data" / "input" / "Thematic_ETF_search.csv")
etf.columns = [c.strip().replace("\n", "") for c in etf.columns]
etf["ticker"] = etf["Ticker"].str.replace(" US Equity", "", regex=False).str.strip()
etf["inception"] = pd.to_datetime(etf["Inception Date"], format="%m/%d/%Y")
INCEPT = etf.set_index("ticker")["inception"].to_dict()

# --- 6 themes: lexicon (term -> regex, annotation only) + theme ETFs from the CSV + milestones ---
THEMES = {
    "space": dict(
        title="Space economy",
        etfs=["ROKT", "UFO", "ARKX", "MARS", "NASA", "WARP"],
        key={"spacex": r"spacex", "starlink": r"starlink", "blue origin": r"blue origin",
             "virgin galactic": r"virgin galactic", "rocket lab": r"rocket lab",
             "rocket/sat launch": r"falcon 9|falcon heavy|starship|rocket launch|satellite launch"},
        milestones=[("Falcon Heavy maiden flight", "2018-02-06"), ("Virgin Galactic NYSE debut", "2019-10-28"),
                    ("Crew Dragon Demo-2", "2020-05-30"), ("Starship first flight", "2023-04-20")]),
    "drones": dict(
        title="Drones & urban air mobility",
        etfs=["ONDL", "BZZ"],
        key={"drone": r"\bdrones?\b", "uav": r"\buavs?\b|unmanned aerial",
             "evtol/air taxi": r"\bevtol\b|air taxi|flying taxi|urban air mobility",
             "joby": r"\bjoby\b", "archer aviation": r"archer aviation", "dji": r"\bdji\b"},
        milestones=[("Amazon Prime Air reveal", "2013-12-01"), ("FAA Part 107 rules", "2016-08-29"),
                    ("Gatwick drone shutdown", "2018-12-19"), ("Joby NYSE debut", "2021-08-11"),
                    ("Ukraine drone war", "2022-02-24")]),
    "crypto": dict(
        title="Crypto & blockchain",
        etfs=["BLOK", "DAPP", "BITQ", "BKCH", "FDIG", "NODE"],
        key={"bitcoin": r"bitcoin", "ethereum": r"ethereum", "blockchain": r"blockchain",
             "crypto": r"\bcrypto", "coinbase": r"coinbase", "stablecoin": r"stablecoin", "nft": r"\bnfts?\b"},
        milestones=[("CME bitcoin futures", "2017-12-17"), ("Coinbase IPO", "2021-04-14"),
                    ("FTX bankruptcy", "2022-11-11"), ("Spot BTC ETFs approved", "2024-01-10")]),
    "esports": dict(
        title="Video games & e-sports",
        etfs=["GAMR", "ESPO", "NERD", "HERO"],
        key={"esports": r"\be-?sports?\b", "video game": r"video ?gam", "twitch": r"\btwitch\b",
             "fortnite": r"fortnite", "epic games": r"epic games", "activision": r"activision",
             "roblox": r"roblox"},
        milestones=[("Amazon buys Twitch", "2014-08-25"), ("Fortnite Battle Royale", "2017-09-26"),
                    ("COVID lockdowns", "2020-03-11"), ("Microsoft bids for Activision", "2022-01-18")]),
    "online_retail": dict(
        title="Online retail / e-commerce",
        etfs=["IBUY", "CLIX", "ONLN", "BUYZ"],
        key={"e-commerce": r"e-?commerce", "online retail/shopping": r"online retail|online shopping|online sales|online store",
             "alibaba": r"alibaba", "shopify": r"shopify", "etsy": r"\betsy\b",
             "shopping events": r"cyber monday|prime day|singles' day|singles day"},
        milestones=[("Alibaba IPO", "2014-09-19"), ("Amazon buys Whole Foods", "2017-06-16"),
                    ("COVID lockdowns", "2020-03-11")]),
    "cyber": dict(
        title="Cybersecurity",
        etfs=["FITE", "BUG", "UCYB", "WCBR"],
        key={"cybersecurity": r"cyber ?security", "cyberattack": r"cyber-? ?attack", "ransomware": r"ransomware",
             "data breach": r"data breach", "hackers": r"\bhack(?:ed|ing|ers?)\b", "malware/phishing": r"malware|phishing"},
        milestones=[("WannaCry", "2017-05-12"), ("Equifax breach", "2017-09-07"), ("SolarWinds hack", "2020-12-13"),
                    ("Colonial Pipeline", "2021-05-07"), ("CrowdStrike outage", "2024-07-19")]),
}
for k, th in THEMES.items():
    th["union"] = re.compile("|".join(th["key"].values()), re.I)
    miss = [t for t in th["etfs"] if t not in INCEPT]
    print(f"{k:14} {len(th['key'])} terms · ETFs: " +
          ", ".join(f"{t} {INCEPT[t].date()}" for t in th["etfs"] if t in INCEPT) +
          (f" · MISSING {miss}" if miss else ""))

In [2]:
# THE corpus — single preprocessed file (ADD-event dating; proof in 0.25, layout in 0.26)
import sys
sys.path.insert(0, str(_ROOT / "scripts"))
import preprocess_news as pp
import polars as pl

CORPUS = pp.ensure()                                         # gate: builds/merges only if missing or stale
df = pl.scan_parquet(CORPUS).select(["Headline", "date"]).collect().to_pandas()
MONTHS = pd.date_range(df.date.min().to_period("M").to_timestamp(),
                       df.date.max().to_period("M").to_timestamp(), freq="MS")
print(f"{len(df):,} distinct headlines · {df.date.min().date()} -> {df.date.max().date()} · {len(MONTHS)} months")

2010: certified add-event-v1 build exists — skipping
2011: certified add-event-v1 build exists — skipping
2012: certified add-event-v1 build exists — skipping
2013: certified add-event-v1 build exists — skipping
2014: certified add-event-v1 build exists — skipping
2015: certified add-event-v1 build exists — skipping
2016: certified add-event-v1 build exists — skipping
2017: certified add-event-v1 build exists — skipping
2018: certified add-event-v1 build exists — skipping
2019: certified add-event-v1 build exists — skipping
2020: certified add-event-v1 build exists — skipping
2021: certified add-event-v1 build exists — skipping
2022: certified add-event-v1 build exists — skipping
2023: certified add-event-v1 build exists — skipping
2024: certified add-event-v1 build exists — skipping
2025: certified add-event-v1 build exists — skipping
corpus up to date: news_corpus.parquet
22,626,656 distinct headlines · 2010-01-01 -> 2025-12-31 · 192 months


In [3]:
# one prefilter pass (union of all 6 lexicons), then per-theme / per-term flags on the small subset
ALL = re.compile("|".join(th["union"].pattern for th in THEMES.values()), re.I)
sub = df[df.Headline.str.contains(ALL, na=False)].copy()
sub["month"] = sub.date.dt.to_period("M").dt.to_timestamp()
print(f"prefilter: {len(sub):,} headlines match at least one theme lexicon\n")

SERIES = {}
for k, th in THEMES.items():
    hits = sub[sub.Headline.str.contains(th["union"], na=False)].copy()
    for name, pat in th["key"].items():
        hits[name] = hits.Headline.str.contains(pat, case=False, regex=True)
    hits["total"] = True
    ts = hits.groupby("month")[["total"] + list(th["key"])].sum().reindex(MONTHS, fill_value=0).astype(int)
    SERIES[k] = ts
    out = OUTPUT_DIR / f"etf_theme_monthly_{k}.parquet"; ts.to_parquet(out)
    print(f"{k:14} {int(ts['total'].sum()):8,} headlines · peak {ts['total'].idxmax():%Y-%m} "
          f"({int(ts['total'].max()):,}/mo) -> {out.name}")

prefilter: 102,148 headlines match at least one theme lexicon

space             7,967 headlines · peak 2025-03 (150/mo) -> etf_theme_monthly_space.parquet
drones            5,957 headlines · peak 2025-09 (123/mo) -> etf_theme_monthly_drones.parquet
crypto           36,199 headlines · peak 2018-01 (1,039/mo) -> etf_theme_monthly_crypto.parquet
esports           7,322 headlines · peak 2023-07 (164/mo) -> etf_theme_monthly_esports.parquet
online_retail    29,860 headlines · peak 2014-09 (625/mo) -> etf_theme_monthly_online_retail.parquet
cyber            15,266 headlines · peak 2011-07 (465/mo) -> etf_theme_monthly_cyber.parquet


In [4]:
# --- one chart per theme: total + key terms · gray dashed = events · red dotted = ETF inceptions ---
PALETTE = ["#2ca02c", "#d62728", "#9467bd", "#ff7f0e", "#8c564b", "#17becf", "#e377c2"]
X_END = pd.Timestamp("2026-07-01")            # extend past corpus end so 2026 ETF launches stay visible
for k, th in THEMES.items():
    ts = SERIES[k]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ts.index, y=ts["total"], name="total", line=dict(width=3, color="#1f77b4")))
    for name, col in zip(th["key"], PALETTE):
        fig.add_trace(go.Scatter(x=ts.index, y=ts[name], name=name, line=dict(width=1.2, color=col)))
    for label, d in th["milestones"]:
        x = pd.Timestamp(d)
        fig.add_shape(type="line", x0=x, x1=x, y0=0, y1=1, yref="paper", line=dict(dash="dash", color="gray", width=1))
        fig.add_annotation(x=x, y=1.0, yref="paper", text=label, textangle=-35, showarrow=False,
                           font=dict(size=8, color="gray"), yshift=8)
    for t in th["etfs"]:
        if t not in INCEPT: continue
        x = INCEPT[t]
        fig.add_shape(type="line", x0=x, x1=x, y0=0, y1=1, yref="paper", line=dict(dash="dot", color="#d62728", width=1.2))
        fig.add_annotation(x=x, y=0.02, yref="paper", text=t, textangle=-90, showarrow=False,
                           font=dict(size=9, color="#d62728"), xshift=-7, yanchor="bottom")
    fig.update_layout(title=f"{th['title']} — keyword headlines / month · ETF inceptions (red) vs news flow",
                      xaxis_title="month", yaxis_title="distinct headlines", template="plotly_white", height=460,
                      xaxis_range=[ts.index.min(), X_END], legend=dict(orientation="h", y=-0.2))
    out = OUTPUT_DIR / f"etf_theme_timeline_{k}.html"; fig.write_html(out)
    print("saved chart ->", out.name)
    fig.show()

saved chart -> etf_theme_timeline_space.html


saved chart -> etf_theme_timeline_drones.html


saved chart -> etf_theme_timeline_crypto.html


saved chart -> etf_theme_timeline_esports.html


saved chart -> etf_theme_timeline_online_retail.html


saved chart -> etf_theme_timeline_cyber.html


In [5]:
# lead/lag: news wave vs ETF launches — does the product (ETF) come before or after the headlines?
print(f"{'theme':14} {'first ETF':>18} {'last ETF':>18} {'news peak':>10} {'peak/mo':>8}   first-seen (top terms)")
for k, th in THEMES.items():
    ts = SERIES[k]; tot = ts["total"]
    launches = sorted((INCEPT[t], t) for t in th["etfs"] if t in INCEPT)
    (d0, t0), (d1, t1) = launches[0], launches[-1]
    firsts = []
    for name in list(th["key"])[:3]:
        nz = ts.index[ts[name] > 0]
        firsts.append(f"{name} {nz.min():%Y-%m}" if len(nz) else f"{name} —")
    print(f"{k:14} {t0 + ' ' + d0.strftime('%Y-%m-%d'):>18} {t1 + ' ' + d1.strftime('%Y-%m-%d'):>18} "
          f"{tot.idxmax().strftime('%Y-%m'):>10} {int(tot.max()):8,}   " + " · ".join(firsts))
print("\nsample headlines around each theme's news peak:")
for k, th in THEMES.items():
    peak = SERIES[k]["total"].idxmax()
    m = sub[(sub.month == peak) & sub.Headline.str.contains(th["union"], na=False)]
    print(f"\n{k} — {peak:%Y-%m}:")
    for h in m.Headline.head(4): print(f"  {h[:100]}")

theme                   first ETF           last ETF  news peak  peak/mo   first-seen (top terms)
space             ROKT 2018-10-23    WARP 2026-05-07    2025-03      150   spacex 2010-03 · starlink 2018-03 · blue origin 2010-02
drones            ONDL 2025-12-30     BZZ 2026-05-05    2025-09      123   drone 2010-01 · uav 2010-01 · evtol/air taxi 2012-07
crypto            BLOK 2018-01-17    NODE 2025-05-14    2018-01    1,039   bitcoin 2011-06 · ethereum 2017-02 · blockchain 2014-02
esports           GAMR 2016-03-09    HERO 2019-10-25    2023-07      164   esports 2010-12 · video game 2010-01 · twitch 2013-08
online_retail     IBUY 2016-04-20    BUYZ 2020-02-27    2014-09      625   e-commerce 2010-01 · online retail/shopping 2010-01 · alibaba 2010-01
cyber             FITE 2017-12-27    WCBR 2021-01-28    2011-07      465   cybersecurity 2010-01 · cyberattack 2010-01 · ransomware 2013-12

sample headlines around each theme's news peak:

space — 2025-03:
  *NASA, SPACEX TARGETS SPHEREX